In [1]:
# ============================================
# SECTION 1: Data Loading & First Inspection
# CHURN PREDICTION PROJECT
# ============================================

import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns
import warnings
#import sklearn
#import tensorflow as tf
warnings.filterwarnings('ignore')

# --- Professional display settings ---
pd.set_option('display.max_columns', None)       # show ALL columns
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places
np.set_printoptions(precision=4, suppress=True)

print("✅ All libraries loaded successfully!")
print(f"Pandas : {pd.__version__}")
print(f"NumPy : {np.__version__}")
#print(f"✅ Matplotlib : {plt.matplotlib.__version__}")
#print(f"✅ Seaborn    : {sns.__version__}")
#print(f"✅ Sklearn    : {sklearn.__version__}")
#print(f"✅ TensorFlow : {tf.__version__}")


✅ All libraries loaded successfully!
Pandas : 3.0.2
NumPy : 2.4.4


In [2]:
# ============================================
# LOAD DATA
# ============================================

df = pd.read_csv('../data/raw/telco_churn.csv')

print("=" * 50)
print("📊 DATASET OVERVIEW")
print("=" * 50)

print(f"\n🔹 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n🔹 Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

📊 DATASET OVERVIEW

🔹 Shape: 7043 rows × 21 columns

🔹 Memory Usage: 7975.08 KB


In [3]:
# ============================================
# PROFESSIONAL DATA AUDIT FUNCTION
# (This is what data scientists actually write)
# ============================================

def data_audit(dataframe):
    """
    Comprehensive first-look audit of any dataset.
    Returns a summary DataFrame — reusable across projects.
    """
    audit = pd.DataFrame({
        'Column'      : dataframe.columns,
        'Data_Type'   : dataframe.dtypes.values,
        'Non_Null'    : dataframe.notnull().sum().values,
        'Null_Count'  : dataframe.isnull().sum().values,
        'Null_%'      : (dataframe.isnull().sum().values / len(dataframe) * 100).round(2),
        'Unique_Values': dataframe.nunique().values,
        'Sample_Value' : [dataframe[col].dropna().iloc[0] if dataframe[col].notnull().any() 
                          else 'ALL NULL' for col in dataframe.columns]
    })
    
    audit = audit.sort_values('Null_%', ascending=False).reset_index(drop=True)
    return audit

audit_report = data_audit(df)
print("\n📋 DATA AUDIT REPORT:")
print(audit_report.to_string(index=False))


📋 DATA AUDIT REPORT:
          Column Data_Type  Non_Null  Null_Count  Null_%  Unique_Values     Sample_Value
      customerID       str      7043           0    0.00           7043       7590-VHVEG
          gender       str      7043           0    0.00              2           Female
   SeniorCitizen     int64      7043           0    0.00              2                0
         Partner       str      7043           0    0.00              2              Yes
      Dependents       str      7043           0    0.00              2               No
          tenure     int64      7043           0    0.00             73                1
    PhoneService       str      7043           0    0.00              2               No
   MultipleLines       str      7043           0    0.00              3 No phone service
 InternetService       str      7043           0    0.00              3              DSL
  OnlineSecurity       str      7043           0    0.00              3               No

In [4]:
# ============================================
# SEPARATE COLUMNS BY TYPE — NUMPY & PANDAS COMBO
# ============================================

# Using numpy to identify numeric columns programmatically
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔢 Numeric Columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\n🔤 Categorical Columns ({len(categorical_cols)}): {categorical_cols}")

# Check target variable distribution
print("\n🎯 TARGET VARIABLE — Churn Distribution:")
churn_counts = df['Churn'].value_counts()
churn_percent = df['Churn'].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage': churn_percent.round(2)
})
print(churn_summary)


🔢 Numeric Columns (3): ['SeniorCitizen', 'tenure', 'MonthlyCharges']

🔤 Categorical Columns (18): ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']

🎯 TARGET VARIABLE — Churn Distribution:
       Count  Percentage
Churn                   
No      5174       73.46
Yes     1869       26.54


In [5]:
# ============================================
# NUMPY STATISTICAL SNAPSHOT
# ============================================

print("\n📐 NUMPY STATISTICAL SNAPSHOT (Numeric Columns):")

numeric_data = df[numeric_cols].values  # Convert to numpy array

stats_report = pd.DataFrame({
    'Column'   : numeric_cols,
    'Mean'     : np.mean(numeric_data, axis=0).round(2),
    'Median'   : np.median(numeric_data, axis=0).round(2),
    'Std_Dev'  : np.std(numeric_data, axis=0).round(2),
    'Min'      : np.min(numeric_data, axis=0),
    'Max'      : np.max(numeric_data, axis=0),
    'Range'    : (np.max(numeric_data, axis=0) - np.min(numeric_data, axis=0))
})

print(stats_report.to_string(index=False))


📐 NUMPY STATISTICAL SNAPSHOT (Numeric Columns):
        Column  Mean  Median  Std_Dev   Min    Max  Range
 SeniorCitizen  0.16    0.00     0.37  0.00   1.00   1.00
        tenure 32.37   29.00    24.56  0.00  72.00  72.00
MonthlyCharges 64.76   70.35    30.09 18.25 118.75 100.50


In [6]:
# ============================================
# SECTION 2: DEEP EDA
# ============================================

import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns

df = pd.read_csv('../data/raw/telco_churn.csv')

# -----------------------------------------------
# PROBLEM: TotalCharges is string, should be float
# -----------------------------------------------

print("Before fix:", df['TotalCharges'].dtype)
print("Sample values:", df['TotalCharges'].head(10).values)

# spaces are hiding inside — classic real world dirty data
# pd.to_numeric with errors='coerce' converts bad values to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("\nAfter fix:", df['TotalCharges'].dtype)
print("Null values created:", df['TotalCharges'].isnull().sum())

Before fix: str
Sample values: <StringArray>
[  '29.85',  '1889.5',  '108.15', '1840.75',  '151.65',   '820.5',  '1949.4',
   '301.9', '3046.05', '3487.95']
Length: 10, dtype: str

After fix: float64
Null values created: 11


In [7]:
# -----------------------------------------------
# WHY ARE THERE NULLS? — Investigate like a detective
# -----------------------------------------------

# Find rows where TotalCharges became null
null_rows = df[df['TotalCharges'].isnull()]

print(f"Rows with null TotalCharges: {len(null_rows)}")
print("\nWhat do these customers look like?")
print(null_rows[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].to_string())

Rows with null TotalCharges: 11

What do these customers look like?
      tenure  MonthlyCharges  TotalCharges Churn
488        0           52.55           NaN    No
753        0           20.25           NaN    No
936        0           80.85           NaN    No
1082       0           25.75           NaN    No
1340       0           56.05           NaN    No
3331       0           19.85           NaN    No
3826       0           25.35           NaN    No
4380       0           20.00           NaN    No
5218       0           19.70           NaN    No
6670       0           73.35           NaN    No
6754       0           61.90           NaN    No


In [8]:
# -----------------------------------------------
# FIX — Fill nulls with 0 (they have 0 tenure = 0 charges)
# -----------------------------------------------

df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Convert Churn to binary number — needed for all analysis
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("✅ Data cleaned!")
print(f"Shape: {df.shape}")
print(f"Churn distribution:\n{df['Churn'].value_counts()}")

✅ Data cleaned!
Shape: (7043, 21)
Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [23]:
# ============================================
# DEEP GROUPBY ANALYSIS — Heavy Pandas
# ============================================

# Separate column types
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                    'PhoneService', 'MultipleLines', 'InternetService',
                    'OnlineSecurity', 'TechSupport', 'Contract',
                    'PaperlessBilling', 'PaymentMethod']

# -----------------------------------------------
# Churn rate by each categorical feature
# -----------------------------------------------

print("=" * 55)
print("📊 CHURN RATE BY CATEGORY")
print("=" * 55)

for col in categorical_cols:
    # groupby + agg together — professional pattern
    churn_by_cat = df.groupby(col)['Churn'].agg(
        Total_Customers = 'count',
        Churned         = 'sum',
        Churn_Rate      = 'mean'
    ).round(3)
    
    churn_by_cat['Churn_Rate_%'] = (churn_by_cat['Churn_Rate'] * 100).round(1)
    churn_by_cat = churn_by_cat.drop('Churn_Rate', axis=1)
    churn_by_cat = churn_by_cat.sort_values('Churn_Rate_%', ascending=False)
    
    print(f"\n🔹 {col}:")
    print(churn_by_cat.to_string())

📊 CHURN RATE BY CATEGORY

🔹 gender:
        Total_Customers  Churned  Churn_Rate_%
gender                                        
Female             3488      939         26.90
Male               3555      930         26.20

🔹 SeniorCitizen:
               Total_Customers  Churned  Churn_Rate_%
SeniorCitizen                                        
1                         1142      476         41.70
0                         5901     1393         23.60

🔹 Partner:
         Total_Customers  Churned  Churn_Rate_%
Partner                                        
No                  3641     1200         33.00
Yes                 3402      669         19.70

🔹 Dependents:
            Total_Customers  Churned  Churn_Rate_%
Dependents                                        
No                     4933     1543         31.30
Yes                    2110      326         15.50

🔹 PhoneService:
              Total_Customers  Churned  Churn_Rate_%
PhoneService                                     

In [19]:
# ── Reload original CSV fresh ──────────────────
df = pd.read_csv('../data/raw/telco_churn.csv')

# ── Check original Churn values before touching ─
print("Original Churn values:")
print(df['Churn'].value_counts(dropna=False))
print("\nRepr check:", [repr(v) for v in df['Churn'].unique()])

Original Churn values:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Repr check: ["'No'", "'Yes'"]


In [20]:
# -----------------------------------------------
# PIVOT TABLE — Like Excel but way more powerful
# -----------------------------------------------
import pandas as pd

print("\n📊 CONTRACT vs INTERNET SERVICE — Churn Pivot Table:")
# Make sure Churn is numeric before pivot
# ── Fix Churn safely ───────────────────────────
df['Churn'] = df['Churn'].astype(str).str.strip()

# Map based on whatever repr showed you above
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

# Verify it worked
print(df['Churn'].dtype)        # should say int64
print(df['Churn'].unique())     # should say [0, 1]

pivot = pd.pivot_table(
    df,
    values  = 'Churn',
    index   = 'Contract',
    columns = 'InternetService',
    aggfunc = 'mean'
).round(3) * 100

print(pivot.to_string())
print("\n💡 Insight: Which contract + internet combo churns most?")


📊 CONTRACT vs INTERNET SERVICE — Churn Pivot Table:
int64
[0 1]
InternetService   DSL  Fiber optic    No
Contract                                
Month-to-month  32.20        54.60 18.90
One year         9.30        19.30  2.50
Two year         1.90         7.20  0.80

💡 Insight: Which contract + internet combo churns most?


In [21]:
# -----------------------------------------------
# FIXED CROSSTAB
# -----------------------------------------------



crosstab = pd.crosstab(
    df['PaymentMethod'],
    df['Churn'],
    normalize='index'   # removed margins=True
).round(3) * 100


# Inspect everything before touching it
print("Shape:", crosstab.shape)
print("Columns:", crosstab.columns.tolist())
print("Index:", crosstab.index.tolist())
print("\nRaw Output:")
print(crosstab)
# Now safely rename — only 2 columns exist
crosstab.columns = ['Not Churned %', 'Churned %']

# Add total customer count as separate column
crosstab['Total_Customers'] = df.groupby('PaymentMethod')['Churn'].count().values

# Sort by churned percentage
crosstab = crosstab.sort_values('Churned %', ascending=False)

print("📊 PAYMENT METHOD vs CHURN:")
print(crosstab.to_string())

Shape: (4, 2)
Columns: [0, 1]
Index: ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']

Raw Output:
Churn                         0     1
PaymentMethod                        
Bank transfer (automatic) 83.30 16.70
Credit card (automatic)   84.80 15.20
Electronic check          54.70 45.30
Mailed check              80.90 19.10
📊 PAYMENT METHOD vs CHURN:
                           Not Churned %  Churned %  Total_Customers
PaymentMethod                                                       
Electronic check                   54.70      45.30             2365
Mailed check                       80.90      19.10             1612
Bank transfer (automatic)          83.30      16.70             1544
Credit card (automatic)            84.80      15.20             1522


In [25]:
# ============================================
# MASTER SETUP CELL — Run this ALWAYS first!
# ============================================
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns

# Load & clean data
df = pd.read_csv('../data/raw/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].astype(str).str.strip().map({'Yes': 1, 'No': 0}).astype(int)

# Define column groups
numeric_cols     = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                    'PhoneService', 'MultipleLines', 'InternetService',
                    'OnlineSecurity', 'TechSupport', 'Contract',
                    'PaperlessBilling', 'PaymentMethod']

print("✅ Data ready!", df.shape)
print("✅ Churn values:", df['Churn'].value_counts().to_dict())
print("✅ Column groups defined!")

✅ Data ready! (7043, 21)
✅ Churn values: {0: 5174, 1: 1869}
✅ Column groups defined!


In [26]:
# ============================================
# NUMPY — IQR OUTLIER DETECTION
# This is used in EVERY real ML project
# ============================================

def detect_outliers_iqr(dataframe, columns):
    """
    Detects outliers using IQR method — pure NumPy.
    Returns summary of outliers per column.
    """
    results = []
    
    for col in columns:
        data = dataframe[col].dropna().values  # numpy array
        
        Q1  = np.percentile(data, 25)
        Q3  = np.percentile(data, 75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers      = data[(data < lower_bound) | (data > upper_bound)]
        outlier_count = len(outliers)
        outlier_pct   = (outlier_count / len(data)) * 100
        
        results.append({
            'Column'        : col,
            'Q1'            : round(Q1, 2),
            'Q3'            : round(Q3, 2),
            'IQR'           : round(IQR, 2),
            'Lower_Bound'   : round(lower_bound, 2),
            'Upper_Bound'   : round(upper_bound, 2),
            'Outlier_Count' : outlier_count,
            'Outlier_%'     : round(outlier_pct, 2)
        })
    
    return pd.DataFrame(results)

outlier_report = detect_outliers_iqr(df, numeric_cols)
print("📊 OUTLIER REPORT:")
print(outlier_report.to_string(index=False))

📊 OUTLIER REPORT:
        Column     Q1      Q3     IQR  Lower_Bound  Upper_Bound  Outlier_Count  Outlier_%
        tenure   9.00   55.00   46.00       -60.00       124.00              0       0.00
MonthlyCharges  35.50   89.85   54.35       -46.02       171.38              0       0.00
  TotalCharges 398.55 3786.60 3388.05     -4683.52      8868.67              0       0.00


In [ ]:
import seaborn as sns

# -----------------------------------------------
# NUMPY — Skewness & Distribution Check
# -----------------------------------------------
import matplotlib.pyplot as plt
print("\n📐 DISTRIBUTION ANALYSIS — NumPy:")

for col in numeric_cols:
    data = df[col].values
    
    mean   = np.mean(data)
    median = np.median(data)
    std    = np.std(data)
    skew   = (3 * (mean - median)) / std   # Pearson skewness formula
    
    # Interpret skewness
    if skew > 0.5:
        skew_type = "Right Skewed ➡️ (high values pulling mean up)"
    elif skew < -0.5:
        skew_type = "Left Skewed ⬅️ (low values pulling mean down)"
    else:
        skew_type = "Roughly Normal ✅"
    
    print(f"\n🔹 {col}:")
    print(f"   Mean={mean:.2f}  Median={median:.2f}  Std={std:.2f}")
    print(f"   Skewness={skew:.3f} → {skew_type}")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# ============================================
# VISUALIZATION DASHBOARD
# ============================================

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('Churn Prediction — EDA Dashboard', 
             fontsize=16, fontweight='bold', y=1.02)

# --- Plot 1: Churn Distribution ---
churn_counts = df['Churn'].value_counts()
axes[0,0].pie(churn_counts, 
              labels    = ['No Churn', 'Churn'],
              autopct   = '%1.1f%%',
              colors    = ['#2ecc71', '#e74c3c'],
              startangle= 90)
axes[0,0].set_title('Overall Churn Distribution')

# --- Plot 2: Tenure Distribution by Churn ---
df[df['Churn']==0]['tenure'].hist(ax=axes[0,1], alpha=0.7, 
                                   color='#2ecc71', label='No Churn', bins=30)
df[df['Churn']==1]['tenure'].hist(ax=axes[0,1], alpha=0.7, 
                                   color='#e74c3c', label='Churn', bins=30)
axes[0,1].set_title('Tenure Distribution by Churn')
axes[0,1].set_xlabel('Tenure (months)')
axes[0,1].legend()

# --- Plot 3: Monthly Charges Boxplot ---
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0,2],
           boxprops    = dict(color='#3498db'),
           medianprops = dict(color='#e74c3c', linewidth=2))
axes[0,2].set_title('Monthly Charges vs Churn')
axes[0,2].set_xlabel('Churn (0=No, 1=Yes)')

# --- Plot 4: Contract Type Bar ---
contract_churn = df.groupby('Contract')['Churn'].mean() * 100
contract_churn.plot(kind='bar', ax=axes[1,0], 
                    color=['#3498db','#e74c3c','#2ecc71'], 
                    edgecolor='black')
axes[1,0].set_title('Churn Rate by Contract Type')
axes[1,0].set_ylabel('Churn Rate %')
axes[1,0].tick_params(axis='x', rotation=15)

# --- Plot 5: Internet Service ---
internet_churn = df.groupby('InternetService')['Churn'].mean() * 100
internet_churn.plot(kind='bar', ax=axes[1,1],
                    color=['#9b59b6','#e74c3c','#2ecc71'],
                    edgecolor='black')
axes[1,1].set_title('Churn Rate by Internet Service')
axes[1,1].set_ylabel('Churn Rate %')
axes[1,1].tick_params(axis='x', rotation=15)

# --- Plot 6: Payment Method ---
pay_churn = df.groupby('PaymentMethod')['Churn'].mean() * 100
pay_churn.plot(kind='barh', ax=axes[1,2], color='#e67e22', edgecolor='black')
axes[1,2].set_title('Churn Rate by Payment Method')
axes[1,2].set_xlabel('Churn Rate %')

# --- Plot 7: Correlation Heatmap (NumPy powered) ---
corr_matrix = df[numeric_cols + ['Churn']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', 
            cmap='RdYlGn', ax=axes[2,0],
            linewidths=0.5)
axes[2,0].set_title('Correlation Heatmap')

# --- Plot 8: Monthly Charges Distribution ---
axes[2,1].hist(df[df['Churn']==0]['MonthlyCharges'], 
               bins=40, alpha=0.7, color='#2ecc71', label='No Churn')
axes[2,1].hist(df[df['Churn']==1]['MonthlyCharges'], 
               bins=40, alpha=0.7, color='#e74c3c', label='Churn')
axes[2,1].set_title('Monthly Charges Distribution')
axes[2,1].set_xlabel('Monthly Charges ($)')
axes[2,1].legend()

# --- Plot 9: Tenure vs Monthly Charges Scatter ---
colors = df['Churn'].map({0: '#2ecc71', 1: '#e74c3c'})
axes[2,2].scatter(df['tenure'], df['MonthlyCharges'], 
                  c=colors, alpha=0.4, s=10)
axes[2,2].set_title('Tenure vs Monthly Charges')
axes[2,2].set_xlabel('Tenure (months)')
axes[2,2].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.savefig('../reports/eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved to reports/eda_dashboard.png")

In [ ]:
# ============================================
# YOUR FINDINGS — Write a professional insights summary
# ============================================

insights = """
📊 KEY BUSINESS INSIGHTS FROM EDA
===================================

1. CHURN RATE:
   - Overall churn rate is ___% — dataset is imbalanced
   - This means model will need special handling (Section 3)

2. CONTRACT TYPE (Biggest Factor):
   - Month-to-month customers churn ___% of the time
   - Two-year contract customers churn only ___% 
   - 🔑 Insight: Lock customers into long contracts!

3. INTERNET SERVICE:
   - Fiber optic customers churn ___% — highest!
   - No internet customers churn only ___%
   - 🔑 Insight: Fiber optic service has quality/price issues

4. PAYMENT METHOD:
   - Electronic check users churn most: ___%
   - Auto-pay customers churn least: ___%
   - 🔑 Insight: Encourage auto-pay enrollment

5. TENURE:
   - New customers (0-12 months) churn most
   - After 2 years, churn drops dramatically
   - 🔑 Insight: First year retention is critical

6. MONTHLY CHARGES:
   - Churned customers pay higher monthly charges on average
   - 🔑 Insight: Price sensitivity is a key churn driver
"""

print(insights)